<a href="https://colab.research.google.com/github/DebStar17/GNSS-Anti-spoofing/blob/main/Hackathon_GNSS_anti_spoofing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Section 1

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

# ==========================================
# CONFIGURATION (Change file names here)
# ==========================================
TRAIN_FILE = 'train.csv'
TEST_FILE = 'train.csv'
SUBMISSION_TEMPLATE = 'submission_format.csv'
OUTPUT_FILE = 'final_submission.csv'

# The 5 independent physics-based features we defined
FEATURES = ['Phase_Divergence', 'Doppler_Delta', 'Corr_Symmetry', 'CN0_Stability', 'Tracking_Error']

# ==========================================
# 1. CORE ENGINE: CLEANUP & PHYSICS MATH
# ==========================================
def load_and_engineer(file_path, is_train=True):
    print(f"--> Loading and cleaning {file_path}...")
    df = pd.read_csv(file_path, low_memory=False)

    # GARBAGE CLEANUP: The first few rows have strings like 'ch0' in numeric columns.
    # This forces 'Carrier_Doppler_hz' to be a number. If it's garbage ('ch0'), it turns into NaN.
    df['Carrier_Doppler_hz'] = pd.to_numeric(df['Carrier_Doppler_hz'], errors='coerce')
    df = df.dropna(subset=['Carrier_Doppler_hz']).copy() # Drop the garbage rows

    # Convert all necessary columns to floats so math works
    cols_to_float = ['Carrier_Doppler_hz', 'Pseudorange_m', 'Carrier_phase',
                     'EC', 'LC', 'PC', 'PIP', 'PQP', 'CN0', 'time']
    for col in cols_to_float:
        df[col] = df[col].astype(float)

    if is_train:
        df['spoofed'] = df['spoofed'].astype(int)

    # Sort by channel then time so our 'Delta' and 'Rolling' math is chronologically correct
    df = df.sort_values(by=['channel', 'time']).reset_index(drop=True)

    print("--> Calculating physically independent features...")
    # Feature 1: Carrier-Phase Divergence (lambda approx 0.19m for L1)
    df['Phase_Divergence'] = df['Pseudorange_m'] - (0.19 * df['Carrier_phase'])

    # Feature 2: Doppler Rate-of-Change (Kinematic acceleration)
    df['Doppler_Delta'] = df.groupby('channel')['Carrier_Doppler_hz'].diff().fillna(0)

    # Feature 3: Correlation Symmetry (Peak distortion)
    epsilon = 1e-9 # Prevents division by zero
    df['Corr_Symmetry'] = (df['EC'] - df['LC']) / (df['EC'] + df['LC'] + epsilon)

    # Feature 4: CN0 Stability (Jitter/Noise rolling standard deviation)
    df['CN0_Stability'] = df.groupby('channel')['CN0'].transform(lambda x: x.rolling(window=5, min_periods=1).std().fillna(0))

    # Feature 5: Tracking Loop Error (Phase Lock deviation)
    df['Tracking_Error'] = np.arctan2(df['PQP'], df['PIP'])

    return df

# ==========================================
# 2. PHASE 1: TRAINING (Finding the t's and c's)
# ==========================================
print("\n--- STARTING TRAINING PHASE ---")
train_df = load_and_engineer(TRAIN_FILE, is_train=True)

X_train = train_df[FEATURES]
y_train = train_df['spoofed']

# The Random Forest acts as our automated optimizer.
# It finds the thresholds (t) and assigns importance weights (c) internally.
print("--> Training model to determine optimal thresholds and weights...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Show the 'c' values (Feature Importances) to prove the physics works
print("\n[Model Weights / Importance (c)]:")
for name, importance in zip(FEATURES, rf_model.feature_importances_):
    print(f" - {name}: {importance:.4f}")

# ==========================================
# 3. PHASE 2: TESTING & GENERATING SUBMISSION
# ==========================================

print("\n--- STARTING TESTING PHASE ---")
test_df = load_and_engineer(TEST_FILE, is_train=False)

X_test = test_df[FEATURES]

print("--> Predicting confidence scores on test data...")
# rf_model.predict_proba outputs [Probability_Clean, Probability_Spoofed]
# We want the Probability_Spoofed (index 1), which acts as our 'Confidence' score
test_df['Confidence'] = rf_model.predict_proba(X_test)[:, 1]

print("\n[Confidence Scores (Per Channel) Statistics]:")
display(test_df['Confidence'].describe())

# IMPORTANT HACKATHON LOGIC:
# The data gives us metrics per 'channel' (satellite), but the submission format
# requires ONE verdict per 'time' step.
# Logic: If ANY channel looks highly spoofed at time T, the whole receiver is spoofed at time T.
print("--> Aggregating channel scores into final receiver verdict...")
submission_agg = test_df.groupby('time')['Confidence'].max().reset_index()

print("\n[Aggregated Confidence Scores (Per Time Step) Statistics]:")
display(submission_agg['Confidence'].describe())

# If the confidence is > 50%, we declare it Spoofed (1). Otherwise, Genuine (0).
submission_agg['Spoofed'] = (submission_agg['Confidence'] > 0.5).astype(int)

# Format to match submission_format.csv exactly
submission_final = submission_agg[['time', 'Spoofed', 'Confidence']]
submission_final['time'] = submission_final['time'].astype(int)

# Save the file
submission_final.to_csv(OUTPUT_FILE, index=False)
print(f"\nSUCCESS! Submission saved to '{OUTPUT_FILE}'. You are ready to upload.")

# Section 2

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. CONFIGURATION
# ==========================================
TRAIN_FILE = 'train.csv'
TEST_FILE = 'train.csv'  # Make sure to run on test.csv for the final output!
OUTPUT_FILE = 'final_hackathon_submission.csv'

# Our highly specific, physics + contextual feature list
FEATURES = [
    'Phase_Divergence', 'Doppler_Delta', 'CN0_Stability',
    'CN0_Drop', 'Corr_Symmetry', 'Tracking_Error',
    'Cliff_Drop', 'Blackout_Intensity'
]

# ==========================================
# 2. THE CLEANUP & "CLIFF" DETECTOR
# ==========================================
def load_and_engineer(file_path, is_train=True):
    print(f"--> Loading & Cleaning {file_path}...")
    df = pd.read_csv(file_path, low_memory=False)

    # --- STRICT GARBAGE REMOVAL ---
    # Convert to numeric; text rows like "ch0" become NaN and are DELETED.
    df['Carrier_Doppler_hz'] = pd.to_numeric(df['Carrier_Doppler_hz'], errors='coerce')
    df = df.dropna(subset=['Carrier_Doppler_hz']).copy()

    # Ensure time is an integer so our grouping works perfectly
    df['time'] = pd.to_numeric(df['time'], errors='coerce').astype(int)

    # Convert everything else to float
    cols = ['Pseudorange_m', 'Carrier_phase', 'EC', 'LC', 'PC', 'PIP', 'PQP', 'CN0']
    for col in cols:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    if is_train:
        df['spoofed'] = pd.to_numeric(df['spoofed'], errors='coerce').fillna(0).astype(int)

    # CRITICAL: Sort by channel AND time before doing any time-math
    df = df.sort_values(by=['channel', 'time']).reset_index(drop=True)

    # --- CONTEXTUAL PHYSICS ---
    # 1. How much did CN0 drop since the last second on THIS specific channel?
    df['CN0_Drop'] = df.groupby('channel')['CN0'].diff().fillna(0)

    # 2. The "Cliff Edge" Detector (Fixed the Channel Bleeding Bug)
    prev_cn0 = df.groupby('channel')['CN0'].shift(1).fillna(0)
    # Trigger 1 if signal is dead (< 5) BUT was healthy a second ago (> 15)
    df['Cliff_Drop'] = ((df['CN0'] < 5) & (prev_cn0 > 15)).astype(int)

    # 3. Mass Blackout Detector (Count how many channels are dead at this exact second)
    df['Is_Dead'] = (df['CN0'] < 5).astype(int)
    df['Blackout_Intensity'] = df.groupby('time')['Is_Dead'].transform('sum')

    # --- STANDARD PHYSICS ---
    df['Phase_Divergence'] = df['Pseudorange_m'] - (0.19 * df['Carrier_phase'])
    df['Doppler_Delta'] = df.groupby('channel')['Carrier_Doppler_hz'].diff().fillna(0)

    epsilon = 1e-9
    df['Corr_Symmetry'] = (df['EC'] - df['LC']) / (df['EC'] + df['LC'] + epsilon)
    df['CN0_Stability'] = df.groupby('channel')['CN0'].transform(lambda x: x.rolling(5, min_periods=1).std().fillna(0))
    df['Tracking_Error'] = np.arctan2(df['PQP'], df['PIP'])

    # Final cleanup of any lingering NaNs from division
    return df.fillna(0)

# ==========================================
# 3. TRAINING THE MODEL
# ==========================================
print("\n--- PHASE 1: BALANCED TRAINING ---")
train_df = load_and_engineer(TRAIN_FILE, is_train=True)

# Using class_weight='balanced' so it pays attention to the rare spoofing rows!
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

print("--> Fitting the Random Forest...")
rf_model.fit(train_df[FEATURES], train_df['spoofed'])

# ==========================================
# 4. TESTING & AGGREGATION
# ==========================================
print("\n--- PHASE 2: SCORING TEST DATA ---")
# IMPORTANT: If you want to verify the 55800-57521 range from your training set,
# temporarily change TEST_FILE to 'train.csv' to see the output.
test_df = load_and_engineer(TEST_FILE, is_train=False)

# Get probabilities
test_df['Confidence'] = rf_model.predict_proba(test_df[FEATURES])[:, 1]

# Aggregate up to the receiver level (1 prediction per time step)
submission_agg = test_df.groupby('time').agg({
    'Confidence': 'max',
    'Blackout_Intensity': 'max',
    'Cliff_Drop': 'max'
}).reset_index()

# OVERRIDE LOGIC: If the model is confident (>0.5) OR if 3+ channels die at once
submission_agg['Spoofed'] = (
    (submission_agg['Confidence'] > 0.5) |
    (submission_agg['Blackout_Intensity'] >= 3)
).astype(int)

# Format for the judges
submission_final = submission_agg[['time', 'Spoofed', 'Confidence']]
submission_final.to_csv(OUTPUT_FILE, index=False)
print(f"\nSUCCESS! Output saved to '{OUTPUT_FILE}'")

# ==========================================
# 5. DIAGNOSTIC VERIFICATION (The 55800 Range)
# ==========================================
print("\n--- DIAGNOSTIC CHECK: The Blackout Range ---")
# Filter the submission for the specific range you mentioned was failing
verify_df = submission_final[(submission_final['time'] >= 55800) & (submission_final['time'] <= 57521)]

if not verify_df.empty:
    spoofed_count = verify_df['Spoofed'].sum()
    total_count = len(verify_df)
    print(f"Time Range: 55800 to 57521")
    print(f"Total Rows in Range: {total_count}")
    print(f"Rows flagged as SPOOFED: {spoofed_count}")
    print(f"Detection Rate: {(spoofed_count/total_count)*100:.2f}%")

    # Show a small sample of what the actual confidence scores look like in that range
    print("\nSample of predictions in this blackout range:")
    print(verify_df.head(5).to_string(index=False))
else:
    print("Range 55800-57521 not found in this dataset (Did you run it on test.csv?).")

# Section 3

In [10]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

# --- CONFIG ---
TRAIN_FILE = 'train.csv'
TEST_FILE = 'train.csv'
OUTPUT_FILE = 'fixed_final_submission.csv'

# Focused features only
FEATURES = ['Phase_Divergence', 'Doppler_Delta', 'CN0_Stability', 'Cliff_Drop', 'Blackout_Intensity', 'CN0']

def load_and_engineer(file_path, is_train=True):
    print(f"--> Loading {file_path}...")
    # Read everything as string first to handle the 'ch0' garbage safely
    df = pd.read_csv(file_path, low_memory=False)

    # 1. DELETE GARBAGE IMMEDIATELY
    # If 'Carrier_Doppler_hz' contains letters, it's a header row. Nuke it.
    df['Carrier_Doppler_hz'] = pd.to_numeric(df['Carrier_Doppler_hz'], errors='coerce')
    df = df.dropna(subset=['Carrier_Doppler_hz']).copy()

    # 2. FORCE NUMERIC
    numeric_cols = ['Pseudorange_m', 'Carrier_phase', 'CN0', 'time']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    if is_train:
        df['spoofed'] = pd.to_numeric(df['spoofed'], errors='coerce').fillna(0).astype(int)

    # 3. SORT BY TIME AND CHANNEL
    df = df.sort_values(by=['channel', 'time']).reset_index(drop=True)

    # 4. ROBUST BLACKOUT FEATURES
    # Compare current CN0 to the average of the last 3 seconds
    df['Prev_CN0_Avg'] = df.groupby('channel')['CN0'].transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean()).fillna(0)

    # Cliff_Drop: Signal is dead (<5) but it WAS healthy (>25)
    df['Cliff_Drop'] = ((df['CN0'] < 5) & (df['Prev_CN0_Avg'] > 25)).astype(int)

    # Blackout_Intensity: Global count per timestamp
    df['Blackout_Intensity'] = df.groupby('time')['Cliff_Drop'].transform('sum')

    # Physics
    df['Phase_Divergence'] = df['Pseudorange_m'] - (0.19 * df['Carrier_phase'])
    df['Doppler_Delta'] = df.groupby('channel')['Carrier_Doppler_hz'].diff().fillna(0)
    df['CN0_Stability'] = df.groupby('channel')['CN0'].transform(lambda x: x.rolling(5).std().fillna(0))

    return df.fillna(0)

# --- EXECUTION ---
train_df = load_and_engineer(TRAIN_FILE)
rf = RandomForestClassifier(n_estimators=150, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(train_df[FEATURES], train_df['spoofed'])

# Print Importance to see if 'Cliff_Drop' is working
print("\n🏆 FEATURE IMPORTANCE:")
for f, imp in zip(FEATURES, rf.feature_importances_):
    print(f"{f}: {imp:.4f}")

# --- TEST SCORING ---
test_df = load_and_engineer(TEST_FILE, is_train=False)
test_df['Confidence'] = rf.predict_proba(test_df[FEATURES])[:, 1]

# AGGREGATION
submission = test_df.groupby('time').agg({'Confidence': 'max', 'Blackout_Intensity': 'max'}).reset_index()

# THE OVERRIDE: If the model is lazy, our Blackout logic takes over
submission['Spoofed'] = ((submission['Confidence'] > 0.6) | (submission['Blackout_Intensity'] >= 2)).astype(int)

submission[['time', 'Spoofed', 'Confidence']].to_csv(OUTPUT_FILE, index=False)
print(f"\n✅ Created {OUTPUT_FILE}")

--> Loading train.csv...

🏆 FEATURE IMPORTANCE:
Phase_Divergence: 0.7527
Doppler_Delta: 0.0217
CN0_Stability: 0.0419
Cliff_Drop: 0.0001
Blackout_Intensity: 0.0005
CN0: 0.1831
--> Loading train.csv...

✅ Created fixed_final_submission.csv
